# Strategy Capacity Lab — research notebook

**Question:** how much capital can an edge absorb before execution costs consume it?

This notebook connects a reproducible return-prediction experiment to an analytical capacity model. Ridge produces a negative result. Positive-alpha calculator scenarios are hypothetical, not fitted claims about this strategy.

Run all cells from the project directory with Python 3.10+. Only the standard library and included dataset are required. No network request or installation is needed. The expanded diagnostics are exploratory: they were designed after the original final-period results were inspected.


In [1]:
from pathlib import Path
from dataclasses import replace
import hashlib
import math
from research import build_result, load_rows
from engine import Scenario, capacity, evaluate

result = build_result()
rows = load_rows()
print(f"{result['raw_n']} source sessions -> {result['n']} supervised observations")
print(f"{result['test_n']} final forecasts, {len(result['folds'])} expanding refits")
print("Input SHA-256:", result['sha256'])
assert hashlib.sha256(Path("data/apple.csv").read_bytes()).hexdigest() == result['sha256']


506 source sessions -> 484 supervised observations
164 final forecasts, 9 expanding refits
Input SHA-256: 24c7604edfd5afe862ddb9f9535e2fd7351f43711bfc442bca52055e15e37bcd


## 1. Establish the information boundary

Features are observed at the preceding close; the target is the following session's open-to-close return. Each refit uses already observed labels and training-only transformations. The penalty is selected using chronological validation, then frozen. Out-of-sample predictions do not make the later diagnostic design preregistered. Choosing a model from these results needs a new independent test.


In [2]:
for name, first, last, count in [
    ("Train", result['train_start'], result['train_end'], result['train_n']),
    ("Validate", result['validation_start'], result['validation_end'], result['validation_n']),
    ("Evaluate", result['test_start'], result['test_end'], result['test_n']),
]:
    print(f"{name:9} {first} -> {last}  {count:3} sessions")
assert all(r['feature_date'] < r['date'] for r in rows)
assert all(f['train_end'] < f['test_start'] for f in result['folds'])
print("Feature/label and refit boundaries verified.")


Train     2015-03-19 -> 2016-03-01  240 sessions
Validate  2016-03-02 -> 2016-06-23   80 sessions
Evaluate  2016-06-24 -> 2017-02-16  164 sessions
Feature/label and refit boundaries verified.


## 2. Inspect regularization before performance

Ridge minimizes mean squared error plus an L2 penalty on standardized coefficients, with an unpenalized intercept. The linear system is solved directly, without gradient descent. The selected penalty is at the top of the predefined grid: evidence for heavy shrinkage within that grid, not global optimality.


In [3]:
print("  penalty     train MSE      valid MSE       coefficient L2")
for item in result['validation_scores']:
    mark = " *" if item['penalty'] == result['selected_penalty'] else ""
    print(f"{item['penalty']:9g}  {item['train_mse']:12.7g}  {item['mse']:12.7g}  {item['coefficient_l2']:12.7g}{mark}")
print("* chosen using validation only")


  penalty     train MSE      valid MSE       coefficient L2
    0.001   0.000229881  0.0001156603   0.004634929
     0.01  0.0002298844  0.0001151436   0.004568084
      0.1  0.0002301241  0.0001111863   0.004025039
        1  0.0002348887  9.952727e-05    0.00199254
       10  0.0002438253  9.207794e-05  0.0003466826
      100  0.0002460606  9.070983e-05  3.769554e-05 *
* chosen using validation only


## 3. Separate forecasting skill from trading results

The position rule is `clip(prediction / 0.01, -1, 1)`, entering at the open and exiting at the close. Both legs pay 3.5 bp. This common mapping does not equalize exposure or risk. Annual net figures are arithmetic and exclude impact, borrowing and financing.

R² is `1 − Σ(y − prediction)² / Σy²`, relative to zero forecasts; it differs from conventional mean-centered R². Positive P&L and improved squared-error forecasting are distinct questions. Momentum is an exploratory comparator, not a newly selected strategy.


In [4]:
print(f"{'Model':20} {'Zero-bmk R2':>12} {'Ann. gross':>12} {'Ann. net':>12} {'Turnover/d':>12}")
for item in result['comparison']:
    m = item['metrics']
    print(f"{item['label']:20} {m['oos_r2']:12.6f} {m['annual_gross']:12.2%} {m['annual_net_before_impact']:12.2%} {m['daily_turnover']:12.2%}")
print("\nRidge fails to beat a zero forecast and loses money before impact.")


Model                 Zero-bmk R2   Ann. gross     Ann. net   Turnover/d
Zero forecast            0.000000        0.00%        0.00%        0.00%
Expanding mean          -0.006060       -0.48%       -0.83%        3.97%
5-day momentum          -0.403637        9.76%        2.51%       82.28%
Ridge regression        -0.006433       -0.52%       -0.87%        3.99%

Ridge fails to beat a zero forecast and loses money before impact.


## 4. Check stability and feature sensitivity

The last fold contains only four observations. Annualizing short-fold results is descriptive scaling, not a year of realized performance. Ablations retain the full-model penalty and refit schedule. Negative ΔMSE means removing a feature improved this evaluation. These are not causal comparisons and do not select a revised model.


In [5]:
print("Fold start    n   Zero-bmk R2    Annual net")
for f in result['folds']:
    print(f"{f['test_start']}  {f['n_test']:3} {f['metrics']['oos_r2']:13.5f} {f['metrics']['annual_net_before_impact']:13.2%}")
print("\nRemoved feature         MSE change     Annual net")
for a in result['ablations']:
    print(f"{a['feature']:22} {a['delta_mse']:13.6g} {a['annual_net_before_impact']:14.2%}")


Fold start    n   Zero-bmk R2    Annual net
2016-06-24   20      -0.05383        -3.77%
2016-07-25   20      -0.01595        -1.94%
2016-08-22   20      -0.00760        -1.53%
2016-09-20   20       0.00159        -0.04%
2016-10-18   20       0.00354         0.34%
2016-11-15   20      -0.01678        -2.72%
2016-12-14   20       0.01327         0.39%
2017-01-13   20       0.03721         1.75%
2017-02-13    4       0.05378         1.93%

Removed feature         MSE change     Annual net
return_1d                -3.0803e-08         -0.83%
momentum_5d             -5.34614e-09         -0.86%
momentum_20d             5.14472e-09         -0.88%
volatility_20d          -9.14671e-09         -0.86%
range_1d                 2.40049e-08         -0.90%
relative_volume         -1.13904e-08         -0.85%


## 5. Quantify conditional uncertainty

Paired daily outcomes and forecast errors are resampled in circular moving blocks, preserving pairing and short runs. Five-session blocks are primary; one and ten sessions test sensitivity. There are 2,000 draws with seed 1729.

The percentile intervals contain no refitting and omit selection, parameter-estimation, survivorship and future-regime uncertainty. They are not posterior probabilities of an edge. The R² interval crosses zero; the net interval is negative under the reference costs.


In [6]:
print("Block   95% interval: annual net       95% interval: zero-benchmark R2")
for u in result['uncertainty']['block_sensitivity']:
    lo, hi = u['annual_net_ci95']
    rlo, rhi = u['oos_r2_ci95']
    print(f"{u['block_length']:5}   [{lo:8.2%}, {hi:8.2%}]          [{rlo: .6f}, {rhi: .6f}]")
print("\nOne-way cost sensitivity, ridge:")
for c in result['cost_sensitivity']:
    print(f"{c['one_way_bps']:4g} bp -> {c['annual_net']: .2%} annual arithmetic net, before impact")


Block   95% interval: annual net       95% interval: zero-benchmark R2
    1   [  -1.66%,   -0.08%]          [-0.014562,  0.002270]
    5   [  -1.76%,   -0.05%]          [-0.015074,  0.002744]
   10   [  -1.81%,   -0.03%]          [-0.017246,  0.002857]

One-way cost sensitivity, ridge:
   0 bp -> -0.52% annual arithmetic net, before impact
   1 bp -> -0.62% annual arithmetic net, before impact
 3.5 bp -> -0.87% annual arithmetic net, before impact
   7 bp -> -1.22% annual arithmetic net, before impact
  15 bp -> -2.02% annual arithmetic net, before impact


## 6. Derive a capital decision

At constant capital A, annual net return is `b − k√A`, where b is delayed alpha less fixed cost. Dollar profit is `bA − kA^(3/2)`; its derivative is `b − (3/2)k√A`. For positive b, the unconstrained optimum is therefore `4/9` of break-even capital, clipped by the daily participation limit. Nonpositive b gives zero optimal positive allocation.

Square-root impact is illustrative. Decay at mean fill delay approximates the average of exponential decay. Overlapping daily orders do not create extra aggregate liquidity. The historical preset averages its inputs; it is not a replay of nonlinear per-session impact.


In [7]:
example = Scenario(participation=1)
c = capacity(example)
print("Hypothetical positive-edge scenario:")
print(f"Break-even capital:    ${c['break_even']:,.0f}")
print(f"Dollar-profit optimum: ${c['optimum']:,.0f}")
print(f"Optimum / break-even:  {c['optimum']/c['break_even']:.6f}")
assert math.isclose(c['optimum']/c['break_even'], 4/9)
assert abs(evaluate(example, c['break_even'])['net']) < 1e-12
m = result['metrics']
historical_summary = Scenario(alpha=m['annual_gross'], daily_vol=m['daily_vol'],
    adv=m['adv'], names=1, turnover=m['daily_turnover'])
print("\nHistorical ridge usable positive capacity:", capacity(historical_summary)['usable'])
assert capacity(historical_summary)['usable'] == 0


Hypothetical positive-edge scenario:
Break-even capital:    $778,596,145
Dollar-profit optimum: $346,042,731
Optimum / break-even:  0.444444

Historical ridge usable positive capacity: 0


## 7. Stanford CS229 mapping

The Stanford-hosted cheatsheets are by Afshine Amidi and Shervine Amidi. This is an educational reference, without affiliation or endorsement.

| Concept | Implementation |
| --- | --- |
| Linear regression and squared loss | Continuous return prediction |
| Normal equations | Penalized linear system; no gradient descent |
| Ridge regularization | Shrunk standardized slopes, free intercept |
| Training / validation / testing | Chronological partitions and expanding refits; no random k-fold |
| Regression diagnostics | MSE and zero-benchmark R², with a different denominator from conventional R² |
| Bias/variance and ablation | Regularization path and fixed-penalty feature removal; not causal attribution |

Training-only scaling, temporal leakage checks, block resampling and capacity mathematics are additional choices. No neural networks, SVMs, tree ensembles, reinforcement learning or clustering are used.

[Supervised Learning](https://web.stanford.edu/~shervine/teaching/cs-229/cheatsheet-supervised-learning/) · [Tips and Tricks](https://web.stanford.edu/~shervine/teaching/cs-229/cheatsheet-machine-learning-tips-and-tricks/)


## Interpretation and next independent experiment

The ridge specification fails the forecast and cost hurdles on this short, single-asset sample. A reproducible negative result prevents allocating hypothetical capacity to an unsupported edge.

A defensible extension freezes choices before testing genuinely new, point-in-time multi-asset data; calibrates spreads and impact; includes borrowing and execution constraints; and reserves an untouched final evaluation. Increasing complexity on the same period would not provide that evidence.

See [README.md](README.md) for complete units, data provenance, assumptions and publication instructions.
